In [30]:
# Load env variables and create client
from dotenv import load_dotenv
from anthropic import Anthropic

load_dotenv()

client = Anthropic()
model = "claude-haiku-4-5"

In [31]:
# Helper functions
def add_user_message(messages, text):
    user_message = {"role": "user", "content": text}
    messages.append(user_message)


def add_assistant_message(messages, text):
    assistant_message = {"role": "assistant", "content": text}
    messages.append(assistant_message)


def chat(messages, system=None, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 1000,
        "messages": messages,
        "stop_sequences": stop_sequences,
    }

    if system:
        params["system"] = system

    message = client.messages.create(**params)
    return message.content[0].text

In [32]:
# Tools and Schemas

from datetime import datetime, timedelta


def add_duration_to_datetime(
    datetime_str, duration=0, unit="days", input_format="%Y-%m-%d"
):
    date = datetime.strptime(datetime_str, input_format)

    if unit == "seconds":
        new_date = date + timedelta(seconds=duration)
    elif unit == "minutes":
        new_date = date + timedelta(minutes=duration)
    elif unit == "hours":
        new_date = date + timedelta(hours=duration)
    elif unit == "days":
        new_date = date + timedelta(days=duration)
    elif unit == "weeks":
        new_date = date + timedelta(weeks=duration)
    elif unit == "months":
        month = date.month + duration
        year = date.year + month // 12
        month = month % 12
        if month == 0:
            month = 12
            year -= 1
        day = min(
            date.day,
            [
                31,
                29 if year % 4 == 0 and (year % 100 != 0 or year % 400 == 0) else 28,
                31,
                30,
                31,
                30,
                31,
                31,
                30,
                31,
                30,
                31,
            ][month - 1],
        )
        new_date = date.replace(year=year, month=month, day=day)
    elif unit == "years":
        new_date = date.replace(year=date.year + duration)
    else:
        raise ValueError(f"Unsupported time unit: {unit}")

    return new_date.strftime("%A, %B %d, %Y %I:%M:%S %p")


def set_reminder(content, timestamp):
    print(f"----\nSetting the following reminder for {timestamp}:\n{content}\n----")


add_duration_to_datetime_schema = {
    "name": "add_duration_to_datetime",
    "description": "Adds a specified duration to a datetime string and returns the resulting datetime in a detailed format. This tool converts an input datetime string to a Python datetime object, adds the specified duration in the requested unit, and returns a formatted string of the resulting datetime. It handles various time units including seconds, minutes, hours, days, weeks, months, and years, with special handling for month and year calculations to account for varying month lengths and leap years. The output is always returned in a detailed format that includes the day of the week, month name, day, year, and time with AM/PM indicator (e.g., 'Thursday, April 03, 2025 10:30:00 AM').",
    "input_schema": {
        "type": "object",
        "properties": {
            "datetime_str": {
                "type": "string",
                "description": "The input datetime string to which the duration will be added. This should be formatted according to the input_format parameter.",
            },
            "duration": {
                "type": "number",
                "description": "The amount of time to add to the datetime. Can be positive (for future dates) or negative (for past dates). Defaults to 0.",
            },
            "unit": {
                "type": "string",
                "description": "The unit of time for the duration. Must be one of: 'seconds', 'minutes', 'hours', 'days', 'weeks', 'months', or 'years'. Defaults to 'days'.",
            },
            "input_format": {
                "type": "string",
                "description": "The format string for parsing the input datetime_str, using Python's strptime format codes. For example, '%Y-%m-%d' for ISO format dates like '2025-04-03'. Defaults to '%Y-%m-%d'.",
            },
        },
        "required": ["datetime_str"],
    },
}

set_reminder_schema = {
    "name": "set_reminder",
    "description": "Creates a timed reminder that will notify the user at the specified time with the provided content. This tool schedules a notification to be delivered to the user at the exact timestamp provided. It should be used when a user wants to be reminded about something specific at a future point in time. The reminder system will store the content and timestamp, then trigger a notification through the user's preferred notification channels (mobile alerts, email, etc.) when the specified time arrives. Reminders are persisted even if the application is closed or the device is restarted. Users can rely on this function for important time-sensitive notifications such as meetings, tasks, medication schedules, or any other time-bound activities.",
    "input_schema": {
        "type": "object",
        "properties": {
            "content": {
                "type": "string",
                "description": "The message text that will be displayed in the reminder notification. This should contain the specific information the user wants to be reminded about, such as 'Take medication', 'Join video call with team', or 'Pay utility bills'.",
            },
            "timestamp": {
                "type": "string",
                "description": "The exact date and time when the reminder should be triggered, formatted as an ISO 8601 timestamp (YYYY-MM-DDTHH:MM:SS) or a Unix timestamp. The system handles all timezone processing internally, ensuring reminders are triggered at the correct time regardless of where the user is located. Users can simply specify the desired time without worrying about timezone configurations.",
            },
        },
        "required": ["content", "timestamp"],
    },
}

batch_tool_schema = {
    "name": "batch_tool",
    "description": "Invoke multiple other tool calls simultaneously",
    "input_schema": {
        "type": "object",
        "properties": {
            "invocations": {
                "type": "array",
                "description": "The tool calls to invoke",
                "items": {
                    "type": "object",
                    "properties": {
                        "name": {
                            "type": "string",
                            "description": "The name of the tool to invoke",
                        },
                        "arguments": {
                            "type": "string",
                            "description": "The arguments to the tool, encoded as a JSON string",
                        },
                    },
                    "required": ["name", "arguments"],
                },
            }
        },
        "required": ["invocations"],
    },
}

pass

In [33]:
from anthropic.types import ToolParam

# get current date time with format
from datetime import datetime

def get_current_datetime(format="%Y-%m-%d %H:%M:%S"):
  # validate format also empty or null values
  if not format:
    raise ValueError("Date format cannot be empty or null")
  try:
    datetime.now().strftime(format)
  except Exception as e:
    raise ValueError(f"Invalid date format: {format}") from e
  return datetime.now().strftime(format)

# test above date function with variaety of formats for differentdate and time formats 
# print(get_current_datetime())
# print(get_current_datetime("%Y-%m-%d"))
# print(get_current_datetime("%H:%M:%S"))
# print(get_current_datetime("%d/%m/%Y %I:%M %p"))
# print(get_current_datetime("%A, %B %d, %Y %I:%M %p"))

# lets add tool schema to be used with anthropic API
from anthropic.types import ToolParam

get_current_datetime_tool: ToolParam = ToolParam({
  "name": "get_current_datetime",
  "description": (
    "Returns the current date and time formatted as a string. "
    "Use this tool when the user asks for the current date, time, or datetime. "
    "The format parameter follows Python strftime conventions "
    "(e.g. '%Y-%m-%d' for date only, '%H:%M:%S' for time only)."
  ),
  "input_schema": {
    "type": "object",
    "properties": {
      "format": {
        "type": "string",
        "description": (
          "Python strftime format string for the output. "
          "Defaults to '%Y-%m-%d %H:%M:%S' (e.g. '2026-09-20 14:30:00'). "
          "Cannot be empty or null. "
          "Examples: '%Y-%m-%d' → '2026-09-20', '%H:%M' → '14:30', "
          "'%B %d, %Y' → 'September 20, 2026'."
        ),
        "default": "%Y-%m-%d %H:%M:%S",
      }
    },
    "required": [],
  },
})

In [34]:
# lets call api and use tool schema we already have message call, setup is already done with api
messages = []
messages.append({
    "role": "user",
    "content": "What is the exact time, formatted as HH:MM:SS?"
})

response = client.messages.create(
    model=model,
    max_tokens=1000,
    messages=messages,
    tools=[get_current_datetime_tool],
)

messages.append({
    "role": "assistant",
    "content": response.content
})




In [35]:
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_018yvEdDkyoeXjLdFwFJKZ9r', caller=DirectCaller(type='direct'), input={'format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}]


In [36]:
# print(response)

#  --- parse and dispatch ---
tool_use_block = next(
    block for block in response.content if block.type == "tool_use"
)

tool_id = tool_use_block.id        # 'toolu_011633RqDEuxsJuS3edJhkwU'
tool_input = tool_use_block.input  # {'format': '%H:%M:%S'}
tool_name = tool_use_block.name    # 'get_current_datetime'

if tool_name == "get_current_datetime":
    result = get_current_datetime(**tool_input)

print(f"tool_id : {tool_id}")
print(f"tool_input : {tool_input}")
print(f"result : {result}")

messages.append({
    "role": "user",
    "content": [{
        "type": "tool_result",
        "tool_use_id": tool_id,
        "content": result,
        "is_error": False
    }]
})

messages

tool_id : toolu_018yvEdDkyoeXjLdFwFJKZ9r
tool_input : {'format': '%H:%M:%S'}
result : 17:30:23


[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'},
 {'role': 'assistant',
  'content': [ToolUseBlock(id='toolu_018yvEdDkyoeXjLdFwFJKZ9r', caller=DirectCaller(type='direct'), input={'format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]},
 {'role': 'user',
  'content': [{'type': 'tool_result',
    'tool_use_id': 'toolu_018yvEdDkyoeXjLdFwFJKZ9r',
    'content': '17:30:23',
    'is_error': False}]}]

In [37]:
print(messages)

[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_018yvEdDkyoeXjLdFwFJKZ9r', caller=DirectCaller(type='direct'), input={'format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_018yvEdDkyoeXjLdFwFJKZ9r', 'content': '17:30:23', 'is_error': False}]}]


In [ ]:
follow_up_response = client.messages.create(
    model=model,
    max_tokens=2000,
    messages=messages,
    tools=[get_current_datetime_tool]
)


follow_up_response : Message(id='msg_011CfEeyNnEZAWoix6miTa3t', container=None, content=[TextBlock(citations=None, text='The exact time is **17:30:23** (5:30:23 PM).', type='text')], model='claude-haiku-4-5-20251001', role='assistant', stop_details=None, stop_reason='end_turn', stop_sequence=None, type='message', usage=Usage(cache_creation=CacheCreation(ephemeral_1h_input_tokens=0, ephemeral_5m_input_tokens=0), cache_creation_input_tokens=0, cache_read_input_tokens=0, inference_geo='not_available', input_tokens=838, output_tokens=23, output_tokens_details=None, server_tool_use=None, service_tier='standard'))
[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_018yvEdDkyoeXjLdFwFJKZ9r', caller=DirectCaller(type='direct'), input={'format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_018yvEdDkyo

In [42]:
stop_reason = follow_up_response.stop_reason
print(stop_reason)  # 'end_turn'


print(messages)

end_turn
[{'role': 'user', 'content': 'What is the exact time, formatted as HH:MM:SS?'}, {'role': 'assistant', 'content': [ToolUseBlock(id='toolu_018yvEdDkyoeXjLdFwFJKZ9r', caller=DirectCaller(type='direct'), input={'format': '%H:%M:%S'}, name='get_current_datetime', type='tool_use', toolset_name=None)]}, {'role': 'user', 'content': [{'type': 'tool_result', 'tool_use_id': 'toolu_018yvEdDkyoeXjLdFwFJKZ9r', 'content': '17:30:23', 'is_error': False}]}, {'role': 'assistant', 'content': [TextBlock(citations=None, text='The exact time is **17:30:23** (5:30:23 PM).', type='text')]}]
